In [ ]:
# 🚀 1. Import all dependencies
import torch
import torchvision.transforms as T
from torchvision.models.segmentation import deeplabv3_resnet101
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Load DeepLabV3 model pretrained on COCO or Cityscapes (we’ll adapt it)
model = deeplabv3_resnet101(pretrained=True).to(device)
model.eval()

# Image transform
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

In [ ]:
# 🚗 2. Load your image and convert it
image_path = r'D:\Master\python-for-dl-homework\week13\redline_images\taxi.png'  # UPDATE THIS
bgr_img = cv2.imread(image_path)
rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
input_tensor = transform(rgb_img).unsqueeze(0).to(device)

In [ ]:
# 🧠 3. Get segmentation output
with torch.no_grad():
    output = model(input_tensor)['out'][0]
    pred = output.argmax(0).cpu().numpy()

# COCO/Cityscapes label maps (check ID if needed)
# Common IDs for Cityscapes: road = 0, car = 13
ROAD_ID = 0
CAR_ID = 13

road_mask = (pred == ROAD_ID).astype(np.uint8)
car_mask = (pred == CAR_ID).astype(np.uint8)

In [ ]:
# 🎯 4. Detect red in HSV — apply only to road area
road_area = cv2.bitwise_and(bgr_img, bgr_img, mask=road_mask)

hsv = cv2.cvtColor(road_area, cv2.COLOR_BGR2HSV)

# Wider red range
lower_red1 = np.array([0, 40, 40])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([160, 40, 40])
upper_red2 = np.array([180, 255, 255])

mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
red_line_mask = cv2.bitwise_or(mask1, mask2)

# Clean up
kernel = np.ones((5, 5), np.uint8)
red_line_mask = cv2.morphologyEx(red_line_mask, cv2.MORPH_OPEN, kernel)

# Optional: keep largest red blob
contours, _ = cv2.findContours(red_line_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if contours:
    max_cnt = max(contours, key=cv2.contourArea)
    red_line_mask = np.zeros_like(red_line_mask)
    cv2.drawContours(red_line_mask, [max_cnt], -1, 255, thickness=cv2.FILLED)


In [ ]:
# 💡 5. Check if car mask and red line mask overlap
intersection = np.logical_and(car_mask, red_line_mask)
violation_pixels = np.sum(intersection)
car_pixels = np.sum(car_mask)

violation_ratio = violation_pixels / car_pixels if car_pixels > 0 else 0
print(f"Red line overlap: {violation_ratio:.3f}")

VIOLATION_THRESHOLD = 0.1  # tune this

if violation_ratio > VIOLATION_THRESHOLD:
    print("🚨 Violation Detected!")
else:
    print("✅ No Violation")


In [ ]:
# 🖼️ 6. Show everything
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.title("Original Image")
plt.imshow(rgb_img)
plt.axis("off")

plt.subplot(1, 4, 2)
plt.title("Road Mask")
plt.imshow(road_mask, cmap='gray')
plt.axis("off")

plt.subplot(1, 4, 3)
plt.title("Car Mask")
plt.imshow(car_mask, cmap='gray')
plt.axis("off")

plt.subplot(1, 4, 4)
plt.title("Red Line (in Road)")
plt.imshow(red_line_mask, cmap='gray')
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load image
image_path = r'D:\Master\python-for-dl-homework\week13\redline_images\taxi.png'
frame = cv2.imread(image_path)
if frame is None:
    raise FileNotFoundError("Image not loaded. Check path.")

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

# Red color in HSV
lower_red1 = np.array([0, 70, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 70, 50])
upper_red2 = np.array([180, 255, 255])

mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
red_mask = cv2.bitwise_or(mask1, mask2)

# Morphological opening to remove small noise
kernel = np.ones((3, 3), np.uint8)
cleaned = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, kernel, iterations=1)

# Contour filtering: keep only large red areas (lines)
min_area = 500  # Adjust this threshold if needed
filtered_mask = np.zeros_like(cleaned)

contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
for cnt in contours:
    area = cv2.contourArea(cnt)
    if area > min_area:
        cv2.drawContours(filtered_mask, [cnt], -1, 255, thickness=cv2.FILLED)

# Apply filtered mask to the original image
red_line_result = cv2.bitwise_and(frame, frame, mask=filtered_mask)
red_line_rgb = cv2.cvtColor(red_line_result, cv2.COLOR_BGR2RGB)

# Show results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Original")
plt.imshow(frame_rgb)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Filtered Red Mask")
plt.imshow(filtered_mask, cmap='gray')
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Detected Red Line (no orientation assumption)")
plt.imshow(red_line_rgb)
plt.axis("off")

plt.tight_layout()
plt.show()